# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their IDs, and associated fields/columns. All referencing is done by `@id` as per Croissant specification.

In [ ]:
# List all record sets, with their IDs and fields

print("Available record sets and their fields (by @id):\n")
record_sets = metadata.recordSet

record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')}")
    print(f"  @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', 'N/A')}")
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    # List fields/columns by @id
    # Croissant can use 'field' or 'column'. Check both
    fields = getattr(rs, 'field', getattr(rs, 'column', []))
    if fields:
        print("  Fields/Columns @id list:")
        for fld in fields:
            fld_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else getattr(fld, '@id', str(fld))
            print(f"    - {fld_id}")
    print("")

print("All RecordSet @id's for extraction:")
print(record_set_ids)

## 3. Data Extraction
Load data from each record set into individual DataFrames for subsequent analysis. Use the record set and field `@id`s shown above.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Reading records from: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample data (first 3 rows):\n{df.head(3)}\n")
    except Exception as e:
        print(f"  Unable to read records or data missing: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering, normalization, and grouping using numeric fields. **All field/column referencing is by `@id`.**

In this example, we pick a numeric field and a group field from the *main* record set. Adjust these IDs to correspond to your dataset.

In [ ]:
# Example RecordSet and field IDs for EDA
# Please adjust if your exploration above found other relevant record sets or field ids.

if len(dataframes) > 0:
    # Use the first non-empty record set
    main_record_set_id = None
    for r in record_set_ids:
        if r in dataframes and dataframes[r].shape[0] > 0:
            main_record_set_id = r
            break

    df = dataframes[main_record_set_id]
    print(f"Main record set for EDA: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")

    # Heuristically select a numeric field for demonstration
    import numpy as np

    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    print(f"Numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]
    else:
        numeric_field = df.columns[0]  # fallback to first, if unknown

    # Optionally pick a group field
    candidate_group_fields = [col for col in df.columns if df[col].dtype == 'object' or str(df[col].dtype).startswith('str')]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
    else:
        group_field = None

    # Filter/normalize/group as in template
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
    filtered_df = df[df[numeric_field] > threshold] if np.issubdtype(df[numeric_field].dtype, np.number) else df
    print(f"Filtered records where {numeric_field} > {threshold} (mean):\n", filtered_df.head())

    # Normalize
    if np.issubdtype(df[numeric_field].dtype, np.number):
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped statistics by {group_field}:")
        print(grouped_df.head())
else:
    print("No record set with data found to demonstrate EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. Example visualization using the first (numeric) field found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field in filtered_df:
    # Histogram
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field exists, show boxplot
    if group_field and group_field in filtered_df:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to discover and analyze tabular data from the FAIR² dataset using only Croissant `@id` referencing. You can now refine this exploration further by selecting different record sets, numeric fields, or grouping keys depending on your research goals.

For more advanced analysis, refer back to the schema and the Croissant documentation. Happy exploring!